In [1]:
from pytao import TaoModel
from pprint import pprint
import json



In [2]:
M = TaoModel('/Users/chrisonian/Code/GitHub/lcls-lattice/bmad/models/sc_sxr/tao.init')

Initialized Tao with /var/folders/wj/lfgr01993dx79p9cm_skykbw0000gn/T/tmpvvodzj_i/tao/tao.init


In [3]:
%%tao
python lat_list 1@0>>Q*|base real:ele.s

-------------------------
Tao> python lat_list 1@0>>Q*|base real:ele.s
-------------------------
Tao> 


In [5]:
SLIST = M.cmd_real('python lat_list -track_only 1@0>>*|model real:ele.s')
LLIST = M.cmd_real('python lat_list -track_only  1@0>>*|model real:ele.l')
NAMES = M.cmd('python lat_list -track_only 1@0>>*|model ele.name')
IXLIST = [i for i in range(len(NAMES))]

ix_of = {}
for ix, name in enumerate(NAMES):
    if name in ix_of:
        pass
#        print("err", name)
    else: 
        ix_of[name] = ix

for s, l, n in zip(SLIST[0:20], LLIST[0:20], NAMES[0:20]):
    print (n, l, s)

BEGINNING 0.0 0.0
BEAM0 0.0 0.0
DCM4B 0.03 0.03
QCM01#1 0.11500000000000002 0.14500000000000002
YCM01 0.0 0.14500000000000002
XCM01 0.0 0.14500000000000002
QCM01#2 0.11499999999999999 0.26
DCM5 0.23686000000000001 0.49686
CM01END 0.0 0.49686
DCMCM1 0.16577 0.66263
HOMCM 0.0 0.66263
DCAP0DA 0.378513 1.041143
ASTRA 0.0 1.041143
DCAP0DB 1.3458670000000001 2.38701
FC1 0.0 2.38701
DMSC0DA 0.0889 2.4759100000000003
BLF1 0.0 2.4759100000000003
DMSC0DB 1.231893 3.707803
VG0H00 0.0 3.707803
DMSC0DC 0.1016 3.809403


In [6]:
# Search for split eles

def find_split_eles(slist, llist, names):
    """
    Searches for split elements
    
    """

    split_eles = {}
    basename = 'xxx'
    split_eles[basename] = {'stubs':[], 'splits':[], 'offsets':[]}
    stubs  = split_eles[basename]['stubs']
    splits = split_eles[basename]['splits']
    offsets = split_eles[basename]['offsets']
    s0 = 0
    for s, l, n in zip(SLIST, LLIST, NAMES):
        
        if l == 0:
            splits.append(n)
            offsets.append(s - s0)
        # Thick ele
        elif n.startswith(basename) and len(n)==len(basename)+1:
            # this is a true split ele     
            stubs.append(n)
            split_eles[basename]['L'] += l
        else:
            # New thick ele
            
            # Removing unwanted eles
            if basename == 'K21_3B':
                # Special case. Leave these
                print(basename, stubs, splits)
                pass
            elif len(stubs) == 1 or len(splits) == 0:
                split_eles.pop(basename)
            # Or if this is obviously a superimpose element
            elif basename.endswith('#'):
                split_eles.pop(basename)
                
            # Try a basename which excludes the final character
            basename = n[:-1]
            split_eles[basename] = {'stubs':[], 'splits':[], 'offsets':[]}
            split_eles[basename]['L'] = l
            s0 = s -l 
            stubs  = split_eles[basename]['stubs']
            splits = split_eles[basename]['splits']
            offsets = split_eles[basename]['offsets']
            stubs.append(n) # add this 
    
        #print (n, l, s, ix)
        
    split_eles.pop('xxx')        
    
    return split_eles

SPLIT_ELES = find_split_eles(SLIST, LLIST, NAMES)
    
pprint(SPLIT_ELES )

{'BCX14': {'L': 0.20354433846537148,
           'offsets': [0.20354433846537745, 0.20354433846537745],
           'splits': ['CNTBC1', 'BC1BCEND'],
           'stubs': ['BCX14A', 'BCX14B']},
 'BCX24': {'L': 0.5491765771623169,
           'offsets': [0.5491765771623136, 0.5491765771623136],
           'splits': ['CNTBC2', 'BC2BCEND'],
           'stubs': ['BCX24A', 'BCX24B']},
 'BCX31B4': {'L': 0.3500135125588256,
             'offsets': [0.3500135125586894],
             'splits': ['CC31BEND'],
             'stubs': ['BCX31B41', 'BCX31B42']},
 'BCX32B4': {'L': 0.3500135125588256,
             'offsets': [0.3500135125586894],
             'splits': ['CC32BEND'],
             'stubs': ['BCX32B41', 'BCX32B42']},
 'BCXDLD4': {'L': 0.3500088889291981,
             'offsets': [0.3500088889292101, 0.3500088889292101],
             'splits': ['CNTDLD', 'CCDLDEND'],
             'stubs': ['BCXDLD4A', 'BCXDLD4B']},
 'BCXDLU4': {'L': 0.3500088889291981,
             'offsets': [0.3500088889292101

# Desplit klystrons

In [7]:
def lines_from_splits(name, split_ele_dict, new_ele_already_in_lattice=False):
    offsets = split_ele_dict['offsets']
    splits = split_ele_dict['splits']
    stubs = split_ele_dict['stubs']
    L = round(split_ele_dict['L'], 9)
    
    

    
    lines = ['!---------------------', f'! {name} LCAVITY']
    if not new_ele_already_in_lattice:
        lines.append(f'{name}: {stubs[0]}, L = {L}')
    lines.append(f'{name}_full: line = ({name})')
    
    if len(splits) == 0:
        lines.append('! does not contain splitting elements. Consider reforming')
    else:
        lines.append('! contains zero length elements:')
    
    for n, o in zip(splits, offsets):
        o = round(o, 9)
        # This is at the end of the ele. Skip.
        if o == L:
            continue
        
        lines.append(f'    {n}[superimpose] = T')
        lines.append(f'    {n}[ref] = {name}')
        lines.append(f'    {n}[ref_origin] = beginning')
        lines.append(f'    {n}[offset] = {o}')
    lines.append('\n')
    
    return lines



SC_LINAC_REPLACEMENTS = {}

for name, ele in SPLIT_ELES.items():
    if any([name.startswith(x) for x in ['CAV'] ]) :
        
        # Strip off ___ from some of these
        name = name.strip('_')
        
        SC_LINAC_REPLACEMENTS[name] = '\n'.join(lines_from_splits(name, ele, new_ele_already_in_lattice=True))
    
    
    
    #line = '\n'.join(lines_from_splits(name, ele))
    
with open('sc_linac_replacements.json', 'w') as outfile:
    json.dump(SC_LINAC_REPLACEMENTS, outfile, ensure_ascii=True, indent='  ')

In [8]:
pprint(SC_LINAC_REPLACEMENTS)

{'CAVC012': '!---------------------\n'
            '! CAVC012 LCAVITY\n'
            'CAVC012_full: line = (CAVC012)\n'
            '! contains zero length elements:\n'
            '    CSPH1[superimpose] = T\n'
            '    CSPH1[ref] = CAVC012\n'
            '    CSPH1[ref_origin] = beginning\n'
            '    CSPH1[offset] = 0.060337187\n'
            '\n',
 'CAVC022': '!---------------------\n'
            '! CAVC022 LCAVITY\n'
            'CAVC022_full: line = (CAVC022)\n'
            '! contains zero length elements:\n'
            '    CSPH2[superimpose] = T\n'
            '    CSPH2[ref] = CAVC022\n'
            '    CSPH2[ref_origin] = beginning\n'
            '    CSPH2[offset] = 0.060337187\n'
            '\n',
 'CAVL025': '!---------------------\n'
            '! CAVL025 LCAVITY\n'
            'CAVL025_full: line = (CAVL025)\n'
            '! contains zero length elements:\n'
            '    CSP02[superimpose] = T\n'
            '    CSP02[ref] = CAVL025\n'
         

In [9]:
BENDS_TO_DESPLIT = [
    'bxh1', 'bxh2', 'bxh3', 'bxh4',
    'bx01', 'bx02', 
    'bx11', 'bx12', 'bx13', 'bx14',
    'bx21', 'bx22', 'bx23', 'bx24']
def desplit_bend_line(name):
    return f'{name}_full: line = ({name})'

BEND_REPLACEMENTS = {}
for name in BENDS_TO_DESPLIT:
    BEND_REPLACEMENTS[name+'_full'] = desplit_bend_line(name)
BEND_REPLACEMENTS 


{'bxh1_full': 'bxh1_full: line = (bxh1)',
 'bxh2_full': 'bxh2_full: line = (bxh2)',
 'bxh3_full': 'bxh3_full: line = (bxh3)',
 'bxh4_full': 'bxh4_full: line = (bxh4)',
 'bx01_full': 'bx01_full: line = (bx01)',
 'bx02_full': 'bx02_full: line = (bx02)',
 'bx11_full': 'bx11_full: line = (bx11)',
 'bx12_full': 'bx12_full: line = (bx12)',
 'bx13_full': 'bx13_full: line = (bx13)',
 'bx14_full': 'bx14_full: line = (bx14)',
 'bx21_full': 'bx21_full: line = (bx21)',
 'bx22_full': 'bx22_full: line = (bx22)',
 'bx23_full': 'bx23_full: line = (bx23)',
 'bx24_full': 'bx24_full: line = (bx24)'}